# 03 — Deep Learning

Train and evaluate deep learning models on F1 telemetry data:
1. Driver classification from anonymous telemetry
2. Lap time prediction from partial telemetry
3. Race outcome prediction

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from src.data import FastF1Loader
from src.data.preprocessing import F1Preprocessor
from src.models.datasets import TelemetryDataset, DriverClassificationDataset
from src.models.driver_classifier import DriverClassifier
from src.models.lap_predictor import LapPredictorCNN, LapPredictorLSTM
from src.models.training import Trainer

sns.set_theme(style='whitegrid')
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Prepare Data

Load telemetry, create distance-normalized windows, and build datasets.

In [ ]:
loader = FastF1Loader()
preprocessor = F1Preprocessor()

# Load a few races for training data
races = [
    (2024, 'Bahrain'), (2024, 'Saudi Arabia'), (2024, 'Australia'),
    (2024, 'Japan'), (2024, 'China'), (2024, 'Miami'),
]

all_windows = []
all_labels = []
all_lap_times = []

for year, gp in races:
    try:
        session = loader.get_session(year, gp, 'R')
        laps = session.laps
        
        for _, lap in laps.iterrows():
            try:
                tel = lap.get_telemetry()
                # Normalize to distance
                tel_norm = preprocessor.normalize_to_distance(tel, num_points=500)
                # Create a single window from the full lap
                channels = ['Speed', 'Throttle', 'Brake', 'nGear', 'RPM', 'DRS']
                available = [c for c in channels if c in tel_norm.columns]
                window = tel_norm[available].values
                
                if window.shape[0] == 500 and not np.isnan(window).any():
                    all_windows.append(window)
                    all_labels.append(lap['Driver'])
                    lt = lap['LapTime'].total_seconds() if hasattr(lap['LapTime'], 'total_seconds') else lap['LapTime']
                    all_lap_times.append(lt)
            except Exception:
                continue
        print(f'Loaded {gp}: {len(all_windows)} total samples')
    except Exception as e:
        print(f'Failed {gp}: {e}')

windows = np.stack(all_windows)
driver_labels = np.array(all_labels)
lap_times = np.array(all_lap_times)

print(f'\nDataset: {windows.shape[0]} laps, {windows.shape[1]} points, {windows.shape[2]} channels')
print(f'Unique drivers: {len(np.unique(driver_labels))}')

## 2. Driver Classification

Can we identify which driver produced a telemetry trace? This tests how
distinguishable driving styles are from telemetry alone.

In [ ]:
# Build dataset
driver_names = driver_labels.tolist()
dataset = DriverClassificationDataset(windows, driver_labels, driver_names)

# Split 80/10/10
n = len(dataset)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
n_test = n - n_train - n_val

train_ds, val_ds, test_ds = torch.utils.data.random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')
print(f'Num drivers: {dataset.num_drivers}')

In [ ]:
# Build model
model = DriverClassifier(
    in_channels=windows.shape[2],
    num_drivers=dataset.num_drivers,
    base_channels=64,
    num_blocks=4,
)

trainer = Trainer(
    model=model,
    criterion=torch.nn.CrossEntropyLoss(),
    device=device,
    experiment_name='driver_classifier',
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Train
history = trainer.fit(train_loader, val_loader, epochs=30, patience=7)

# Plot training curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history['train_loss'], label='Train Loss')
ax.plot(history['val_loss'], label='Val Loss')
ax.axvline(history['best_epoch'] - 1, color='red', linestyle='--', label=f'Best epoch ({history["best_epoch"]})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Driver Classifier Training')
ax.legend()
plt.show()

In [ ]:
# Evaluate on test set — confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

trainer.load_checkpoint('driver_classifier_best.pt')
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        preds = model.predict(X).cpu()
        all_preds.extend(preds.numpy())
        all_true.extend(y.numpy())

all_preds = np.array(all_preds)
all_true = np.array(all_true)

# Map indices back to driver names
idx_to_driver = dataset.idx_to_driver
driver_names_sorted = [idx_to_driver[i] for i in range(dataset.num_drivers)]

cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, xticklabels=driver_names_sorted, yticklabels=driver_names_sorted,
            annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Driver Classification Confusion Matrix')
plt.show()

accuracy = (all_preds == all_true).mean()
print(f'\nTest accuracy: {accuracy:.1%}')
print(f'\nClassification report:')
print(classification_report(all_true, all_preds, target_names=driver_names_sorted))

## 3. Lap Time Prediction

Predict total lap time from the first 50% of the telemetry trace.

In [ ]:
# Build dataset with partial telemetry
lap_dataset = TelemetryDataset.from_preprocessed(
    windows, lap_times, fraction=0.5
)

n = len(lap_dataset)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
n_test = n - n_train - n_val

train_ds, val_ds, test_ds = torch.utils.data.random_split(
    lap_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)

print(f'Partial telemetry shape: {lap_dataset.windows.shape}')

In [ ]:
# Train CNN model
cnn_model = LapPredictorCNN(in_channels=windows.shape[2])
trainer = Trainer(
    model=cnn_model,
    criterion=torch.nn.MSELoss(),
    device=device,
    experiment_name='lap_predictor_cnn',
)

history = trainer.fit(train_loader, val_loader, epochs=30, patience=7)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history['train_loss'], label='Train')
ax.plot(history['val_loss'], label='Val')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Lap Time Predictor (CNN)')
ax.legend()
plt.show()

In [ ]:
# Evaluate
trainer.load_checkpoint('lap_predictor_cnn_best.pt')
predictions = trainer.predict(test_loader)
actuals = np.array([y.numpy() for _, y in test_ds])

mae = np.abs(predictions - actuals).mean()
rmse = np.sqrt(((predictions - actuals) ** 2).mean())
print(f'Test MAE: {mae:.3f}s')
print(f'Test RMSE: {rmse:.3f}s')

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(actuals, predictions, alpha=0.3, s=10)
lims = [min(actuals.min(), predictions.min()), max(actuals.max(), predictions.max())]
ax.plot(lims, lims, 'r--', label='Perfect')
ax.set_xlabel('Actual Lap Time (s)')
ax.set_ylabel('Predicted Lap Time (s)')
ax.set_title(f'Lap Time Prediction (MAE={mae:.3f}s)')
ax.legend()
plt.show()

## 4. Next Steps

- Try LSTM and Transformer architectures for lap prediction
- Experiment with different telemetry fractions (25%, 50%, 75%)
- Build race outcome prediction model (MLP on aggregated features)
- Compare XGBoost baseline vs neural network for race prediction